## Image references & tagging in practice

Recall the reference grammar from module 02, now with what each piece actually controls:

```
[registry-host[:port]/][namespace/]repository[:tag][@digest]
```

| Piece | Default | Determines |
|---|---|---|
| `registry-host` | `docker.io` | which registry's API to hit |
| `namespace` | `library` (on Hub) | which user/org owns it |
| `repository` | required | the image name |
| `tag` | `latest` | which manifest the registry points to *right now* — **mutable** |
| `@digest` | none | a specific manifest — **immutable**; wins over the tag if both are present |

**Tag + digest together** is the production pattern: the tag for humans, the digest for certainty — `myapp:1.4.2@sha256:…` reads clearly *and* resolves to exact bytes.

**Tagging in practice.** In a real org, CI tags one freshly-built image several ways at once — the **git SHA** (`myapp:a1b2c3d`), a **semver** if the commit is a release (`myapp:1.4.2`), and a **channel** (`myapp:main`, `:rc`). All point at the same manifest digest, so pushing all of them costs one upload. At deploy time the manifest references the image by **digest** (or an immutable semver) for reproducibility.

**Is `:latest` ever OK?** Three answers: for images **you publish** — almost never (downstream users get a moving target); for images **you consume locally** — fine (`docker run nginx` saves typing); for **production deploys** — never (a `docker pull` six months on won't necessarily get the image you tested). Some registries enforce this with **tag immutability** (ECR's "immutable tags", Harbor's "immutable repositories") — turn it on for release tags so a `1.4.2` can never be silently re-pointed. The rule: **name for humans with tags, deploy for certainty with digests.**